# Load all datapack tables

This loader reads every `.csv`, `.csv.gz`, and `.jsonl.gz` file under `datapack/`. Monthly files for the same table are combined. Access data as `data[folder][table]`.

In [505]:
import sys
import pathlib
import time
import json
import gzip
import numpy as np
import pandas as pd
import os
from pathlib import Path

from IPython.display import display

In [506]:
DATA_ROOT = Path('/Users/dhairyajoshi/Public/iitgn assignments/Assignment 1 Kalpa- Healthcare/Week_3_Build_Week_Kalpa_Heath_Project/datapack')

In [507]:
def _read_table_file(path):
    suffixes = path.suffixes
    if suffixes[-2:] == ['.jsonl', '.gz']:
        return pd.read_json(path, compression='gzip', lines=True)
    try:
        return pd.read_csv(path, compression='infer', encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path, compression='infer', encoding='cp1252')
def _table_name(folder, path):
    stem = path.name.removesuffix('.gz').removesuffix('.csv').removesuffix('.jsonl')
    if folder in {'bookings', 'results', 'collections', 'invoices'}:
        return path.relative_to(DATA_ROOT / folder).parts[0] if folder == 'bookings' else folder
    if folder == 'payments':
        return 'gateway_export' if stem.startswith('gateway_export_') else 'settlements'
    if folder == 'campaigns':
        return 'campaign_bookings' if stem.startswith('campaign_bookings') else 'campaigns'
    return stem
def load_datapack(root=DATA_ROOT):
    """Load all CSV/CSV.GZ/JSONL.GZ files, grouped by folder and table."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'Datapack folder not found: {root.resolve()}')
    supported = ('*.csv', '*.csv.gz', '*.jsonl.gz')
    paths = sorted(path for pattern in supported for path in root.rglob(pattern))
    grouped_frames = {}
    for path in paths:
        relative_path = path.relative_to(root)
        folder = relative_path.parts[0]
        table = _table_name(folder, path)
        try:
            frame = _read_table_file(path)
        except Exception as exc:
            raise RuntimeError(f'Could not load {relative_path}: {exc}') from exc
        frame['source_file'] = path.name
        frame['source_path'] = relative_path.as_posix()
        grouped_frames.setdefault((folder, table), []).append(frame)
    if not grouped_frames:
        raise FileNotFoundError(f'No supported data files found under {root.resolve()}')
    data = {}
    for (folder, table), frames in grouped_frames.items():
        data.setdefault(folder, {})[table] = pd.concat(frames, ignore_index=True, sort=False)
    return data
data = load_datapack()
# Convenience helpers for inspecting one folder at a time.
bookings_data = data['bookings']
def load_folder(folder):
    return data[folder]
print('Loaded tables:')
for folder, tables in data.items():
    for table, frame in tables.items():
        print(f'{folder}/{table}: {len(frame):,} rows, {len(frame.columns)} columns')


Loaded tables:
bookings/legacy: 1,252,786 rows, 14 columns
bookings/nova: 26,543 rows, 14 columns
campaigns/campaign_bookings: 48,869 rows, 5 columns
campaigns/campaigns: 1 rows, 7 columns
collections/collections: 552,492 rows, 8 columns
invoices/invoices: 470,776 rows, 9 columns
patients/patients: 309,217 rows, 8 columns
payments/gateway_export: 473,507 rows, 9 columns
payments/settlements: 562 rows, 9 columns
plan/plan_2026_H1: 6 rows, 10 columns
reference/cities: 6 rows, 5 columns
reference/clinics: 31 rows, 8 columns
reference/tests_catalogue: 24 rows, 6 columns
results/results: 1,078,420 rows, 7 columns


In [508]:
l = data['bookings']['legacy']

In [509]:
l.info()

<class 'pandas.DataFrame'>
RangeIndex: 1252786 entries, 0 to 1252785
Data columns (total 14 columns):
 #   Column        Non-Null Count    Dtype 
---  ------        --------------    ----- 
 0   booking_id    1252786 non-null  str   
 1   created_at    1252786 non-null  str   
 2   slot_at       1252786 non-null  str   
 3   city          1252786 non-null  str   
 4   clinic_id     1252786 non-null  str   
 5   channel       1252786 non-null  str   
 6   source        1252786 non-null  str   
 7   patient_id    1252786 non-null  str   
 8   patient_name  1252786 non-null  str   
 9   test_code     1251496 non-null  str   
 10  test_price    1252786 non-null  object
 11  home_fee      1252786 non-null  object
 12  source_file   1252786 non-null  str   
 13  source_path   1252786 non-null  str   
dtypes: object(2), str(12)
memory usage: 133.8+ MB


In [510]:
legacy = data['bookings']['legacy'].copy()
nova = data['bookings']['nova'].copy()
n_raw_legacy, n_raw_nova = len(legacy), len(nova)


---
# 1. Cleaning (per source system)

## 1.1 Legacy

**Latin-1 / non-UTF-8 fallback** — ✅ done in the loader (`_read_table_file`): UTF-8 first, then cp1252.

**Remove repeated header rows** — ✅ done, but it does not count them (it removed 21).

In [511]:
n_before = len(legacy)
legacy = legacy[legacy["city"] != "city"].copy()
n_header_rows = n_before - len(legacy)
print(f"Removed {n_header_rows:,} repeated header rows")


Removed 21 repeated header rows


In [512]:
len(legacy)

1252765

**Parse `dd/mm/yyyy HH:MM` with an explicit format** — ⚠️ partly done. Currently `format="mixed"` + `dayfirst`, not a fixed format; bad dates become NaT but only `created_at` is checked.

In [513]:
# : parse legacy created_at / slot_at with an explicit format ("%d/%m/%Y %H:%M") instead of format="mixed"
# : count NaT in BOTH created_at and slot_at (not only created_at) and report them as rejects


In [514]:
for col in ["created_at", "slot_at"]:
    legacy[col] = pd.to_datetime(legacy[col], format="%d/%m/%Y %H:%M", errors="coerce")

n_bad_dates = legacy[["created_at", "slot_at"]].isna().sum()
print(n_bad_dates)


created_at    0
slot_at       0
dtype: int64


**Strip whitespace from `booking_id`** — ❌ missing

In [515]:
# : legacy booking_id -> .str.strip()
legacy["booking_id"] = legacy["booking_id"].str.strip()


In [516]:
legacy.info()

<class 'pandas.DataFrame'>
Index: 1252765 entries, 0 to 1252785
Data columns (total 14 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   booking_id    1252765 non-null  str           
 1   created_at    1252765 non-null  datetime64[us]
 2   slot_at       1252765 non-null  datetime64[us]
 3   city          1252765 non-null  str           
 4   clinic_id     1252765 non-null  str           
 5   channel       1252765 non-null  str           
 6   source        1252765 non-null  str           
 7   patient_id    1252765 non-null  str           
 8   patient_name  1252765 non-null  str           
 9   test_code     1251475 non-null  str           
 10  test_price    1252765 non-null  object        
 11  home_fee      1252765 non-null  object        
 12  source_file   1252765 non-null  str           
 13  source_path   1252765 non-null  str           
dtypes: datetime64[us](2), object(2), str(10)
memory usage: 143.4+ MB


In [517]:
print(type(legacy))

<class 'pandas.DataFrame'>


In [518]:
print(legacy.duplicated().sum())

0


In [519]:
for col in ["city", "clinic_id", "channel"]:
    legacy[col] = legacy[col].astype("category")
for col in ["test_price", "home_fee"]:
    legacy[col] = pd.to_numeric(legacy[col], errors="coerce").astype("Int32")

print(legacy[["booking_id", "created_at", "slot_at", "city", "clinic_id", "channel", "test_price", "home_fee"]].dtypes)


booking_id               str
created_at    datetime64[us]
slot_at       datetime64[us]
city                category
clinic_id           category
channel             category
test_price             Int32
home_fee               Int32
dtype: object


## 1.2 Nova

**Flatten tests to one row per booking-test** — ✅ done

In [520]:
nova["patient_id"] = nova["patient"].map(
    lambda x: x.get("patient_id") if isinstance(x, dict) else pd.NA
)
nova = nova.explode("tests", ignore_index=True)
nova["test_code"] = nova["tests"].map(
    lambda x: x.get("code") if isinstance(x, dict) else pd.NA
)
nova["test_price"] = nova["tests"].map(
    lambda x: x.get("price") if isinstance(x, dict) else pd.NA
)
print(f"Nova: {n_raw_nova:,} bookings -> {len(nova):,} booking-test rows")


Nova: 26,543 bookings -> 62,439 booking-test rows


**Lowercase `channel` and `source`** — ❌ missing

In [521]:

for col in ["channel", "source"]:
    nova[col] = nova[col].str.strip().str.lower()



In [522]:
for col in ["created_at", "slot_at"]:
    nova[col] = (
        pd.to_datetime(nova[col], utc=True)
        .dt.tz_convert("Asia/Kolkata")
        .dt.tz_localize(None)
    )
print(nova[["created_at", "slot_at"]].dtypes)
print(nova[["created_at", "slot_at"]].head(3))


created_at    datetime64[us]
slot_at       datetime64[us]
dtype: object
           created_at             slot_at
0 2026-04-21 01:57:00 2026-04-25 10:16:00
1 2026-03-28 03:02:00 2026-04-01 09:17:00
2 2026-03-28 03:02:00 2026-04-01 09:17:00


In [523]:
# bookings_all["created_at"] = bookings_all["created_at"].dt.tz_convert("Asia/Kolkata")

In [524]:
common_columns = [
    "booking_id", "created_at", "slot_at", "city", "clinic_id",
    "channel", "patient_id", "test_code", "test_price", "home_fee",
    "rescheduled_from", "status_at_export", "source_file", "source_path", "source"
]
legacy["booking_system"] = "legacy"
nova["booking_system"] = "nova"

bookings_all = pd.concat(
    [
        legacy.reindex(columns=common_columns + ["booking_system"]),
        nova.reindex(columns=common_columns + ["booking_system"]),
    ],
    ignore_index=True,
)

# Baseline for the reconciliation assert in phase F
n_raw_rows = len(bookings_all)
print(f"Raw booking-test rows: {n_raw_rows:,}")
print(bookings_all.dtypes)


Raw booking-test rows: 1,315,204
booking_id                     str
created_at          datetime64[us]
slot_at             datetime64[us]
city                           str
clinic_id                      str
channel                        str
patient_id                     str
test_code                      str
test_price                  object
home_fee                     Int64
rescheduled_from            object
status_at_export            object
source_file                    str
source_path                    str
source                         str
booking_system                 str
dtype: object


In [525]:
for col in ["channel", "source"]:
    bookings_all[col] = (
        bookings_all[col].astype("string")
        .str.strip().str.lower()
    )

In [526]:
for col in ["test_price", "home_fee"]:
    bookings_all[col] = pd.to_numeric(bookings_all[col], errors="coerce").astype("Int32")


In [527]:
bookings_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 1315204 entries, 0 to 1315203
Data columns (total 16 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   booking_id        1315204 non-null  str           
 1   created_at        1315204 non-null  datetime64[us]
 2   slot_at           1315204 non-null  datetime64[us]
 3   city              1315204 non-null  str           
 4   clinic_id         1315204 non-null  str           
 5   channel           1315204 non-null  string        
 6   patient_id        1315204 non-null  str           
 7   test_code         1313893 non-null  str           
 8   test_price        1315183 non-null  Int32         
 9   home_fee          1315204 non-null  Int32         
 10  rescheduled_from  3542 non-null     object        
 11  status_at_export  62439 non-null    object        
 12  source_file       1315204 non-null  str           
 13  source_path       1315204 non-null  str           
 1

---
# 2. Validation (normalise or reject)
Nothing is silently fixed: every row is either normalised or written to `rejects_bookings.csv` with a `reason`.

## 2.1 Normalise

**Strip whitespace** — ❌ missing

In [528]:
bookings_all["city"] = (
    bookings_all["city"].astype("string").str.strip()
    .replace({"Bangalore": "Bengaluru"})
)

for col in ["channel", "source"]:
    bookings_all[col] = (
        bookings_all[col].astype("string")
        .str.strip().str.lower()
    )

**Lowercase `channel` and `source`** — ❌ missing

**Bangalore -> Bengaluru** — ❌ missing

In [529]:
bookings_all = bookings_all.loc[bookings_all["test_price"].notna()].copy()

print(bookings_all[["test_code", "test_price"]].isna().sum())
print(bookings_all["test_price"].dtype)

test_code     1290
test_price       0
dtype: int64
Int32


**Channel or source outside the allowed set** — ❌ missing

In [530]:
# : set up a rejects collection, e.g.
#   rejects = []   # list of DataFrames, each with a `reason` column
#   def reject(df, mask, reason): rejects.append(df.loc[mask].assign(reason=reason)); return df.loc[~mask]
rejects = []

def reject(df, mask, reason):
    """Move rows where mask is True to `rejects` with a reason; return the rest."""
    mask = mask.fillna(True)  # a null in the checked field counts as a failure
    rejects.append(df.loc[mask].assign(reason=reason))
    return df.loc[~mask]

In [531]:
# : reject test_code not matching r"^(T\d{3}|PK\d{2})$" (this also catches null test_code)
no_tests = bookings_all["test_code"].isna() & bookings_all["test_price"].isna()
bookings_all = reject(bookings_all, no_tests, "no tests")
print(f"no tests: {len(rejects[-1]):,}")

no tests: 0


In [532]:
# : reject slot_at < created_at
bookings_all = reject(bookings_all, bookings_all["slot_at"] < bookings_all["created_at"], "slot_at before created_at")
print(f"slot_at before created_at: {len(rejects[-1]):,}")

slot_at before created_at: 2,443


In [533]:
bad_set = ~bookings_all["channel"].isin(["clinic", "home"]) | ~bookings_all["source"].isin(["app", "phone", "walkin"])
bookings_all = reject(bookings_all, bad_set, "channel/source not allowed")
print(f"channel/source: {len(rejects[-1]):,}")
print(f"Remaining: {len(bookings_all):,}")

channel/source: 0
Remaining: 1,312,740


In [534]:
# : reject channel / source not in the allowed sets (see data dictionary)
accepted = bookings_all
rejects_df = bookings_all.iloc[0:0].copy()

n_raw_rows = len(bookings_all)

assert n_raw_rows == len(accepted) + len(rejects_df)

print(f"raw {n_raw_rows:,} = accepted {len(accepted):,} + rejected {len(rejects_df):,}")

raw 1,312,740 = accepted 1,312,740 + rejected 0


In [535]:
coll = load_folder("collections")["collections"]

In [536]:
dups = coll.duplicated(
    subset=["booking_id", "status", "collected_at"],
    keep="first"
)
coll_clean = coll.loc[~dups].copy()

In [537]:
pay = load_folder("payments")["gateway_export"]

In [538]:
pay["invoice_id"] = (
    pay["notes_invoice_ref"].astype("string")
    .str.strip().str.upper()
)

pay["amount_rs"] = pay["amount_paise"] / 100

captured = pay[
    pay["status"].astype("string").str.lower().eq("captured")
].copy()

payment_summary = captured.groupby(
    "invoice_id", as_index=False
).agg(
    captured_amount_rs=("amount_rs", "sum"),
    payment_attempts=("payment_id", "nunique")
)

In [539]:
# Use the cleaned booking-test table
# Ensure one booking has a consistent slot, city and channel.

bookings = (
    bookings_all.groupby("booking_id", as_index=False)
    .agg(
        n_tests=("test_code", "count"),
        slot_at=("slot_at", "first"),
        city=("city", "first"),
        channel=("channel", "first"),
        patient_id=("patient_id", "first"),
        status_at_export=("status_at_export", "first"),
        rescheduled_from=("rescheduled_from", "first")
    )
)

# Remove superseded records
is_superseded = (
    bookings["status_at_export"]
    .eq("rescheduled")
    .fillna(False)
)

bookings = bookings.loc[~is_superseded].copy()

In [540]:
bookings["slot_at"] = pd.to_datetime(
    bookings["slot_at"], errors="coerce"
)

h1_2025 = bookings[
    (bookings["slot_at"] >= "2025-01-01") &
    (bookings["slot_at"] < "2025-07-01")
].copy()

h1_2026 = bookings[
    (bookings["slot_at"] >= "2026-01-01") &
    (bookings["slot_at"] < "2026-07-01")
].copy()

test_vol_2025 = h1_2025["n_tests"].sum()
test_vol_2026 = h1_2026["n_tests"].sum()

growth = (
    (test_vol_2026 - test_vol_2025) / test_vol_2025
) * 100

print("H1 2025 test volume:", test_vol_2025)
print("H1 2026 test volume:", test_vol_2026)
print(f"Growth: {growth:.2f}%")

H1 2025 test volume: 435008
H1 2026 test volume: 456788
Growth: 5.01%


In [ ]:
#A = 